
# Phi-2 Retail SQL Fine-Tuning — Before/After Benchmark

## Why phi-2 instead of phi-1_5?
> `phi-1_5` uses an old MixFormer architecture with `model_type: phi-msft`. On Python 3.12 with modern transformers it throws `AttributeError: 'PhiConfig' has no attribute 'pad_token_id'` regardless of version pinning. `phi-2` is natively integrated in transformers and works with all modern libraries.

## Setup
1. **Runtime → Change runtime type → T4 GPU**
2. Run **Cell 1** only → **Runtime → Restart session**
3. After restart, run **Cell 2 onwards** top to bottom

| | Detail |
|--|--------|
| Model | `microsoft/phi-2` (2.7B, fp16) |
| Libraries | Latest transformers + peft + datasets (no TRL needed) |
| Schema | 3 tables: `dim_product`, `dim_store`, `fact_sales` |
| Training | 35 Q&A pairs, 10 epochs |
| Test | 12 questions — same before and after training |
| VRAM | ~8-10GB on T4 ✅ |

## CELL 1: Install

In [ ]:
# Run this cell, then RESTART (Runtime → Restart session).
# Do NOT add any imports here. Do NOT re-run after restart.
!pip install -q -U transformers peft datasets accelerate sacrebleu torchao>=0.16.0
print("✅ Done — RESTART NOW: Runtime → Restart session → then run Cell 2 onwards")

✅ Done — RESTART NOW: Runtime → Restart session → then run Cell 2 onwards


## CELL 2: Imports

In [ ]:
import re, gc, torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType
import sacrebleu

print("Imports OK ✅")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ No GPU — switch to T4 GPU runtime'}")

Imports OK ✅
GPU: Tesla T4


## CELL 3: Schema

In [ ]:
SCHEMA = """CREATE TABLE dim_product (
    product_id   INTEGER PRIMARY KEY,
    product_name VARCHAR(100),
    category     VARCHAR(50),
    brand        VARCHAR(50),
    unit_price   FLOAT
);
CREATE TABLE dim_store (
    store_id     INTEGER PRIMARY KEY,
    store_name   VARCHAR(100),
    city         VARCHAR(50),
    region       VARCHAR(50),
    store_format VARCHAR(20)
);
CREATE TABLE fact_sales (
    sale_id    INTEGER PRIMARY KEY,
    product_id INTEGER REFERENCES dim_product(product_id),
    store_id   INTEGER REFERENCES dim_store(store_id),
    sale_date  DATE,
    quantity   INTEGER,
    revenue    FLOAT,
    cost       FLOAT,
    discount   FLOAT
);"""

print("Schema ✅")
print(SCHEMA)

Schema ✅
CREATE TABLE dim_product (
    product_id   INTEGER PRIMARY KEY,
    product_name VARCHAR(100),
    category     VARCHAR(50),
    brand        VARCHAR(50),
    unit_price   FLOAT
);
CREATE TABLE dim_store (
    store_id     INTEGER PRIMARY KEY,
    store_name   VARCHAR(100),
    city         VARCHAR(50),
    region       VARCHAR(50),
    store_format VARCHAR(20)
);
CREATE TABLE fact_sales (
    sale_id    INTEGER PRIMARY KEY,
    product_id INTEGER REFERENCES dim_product(product_id),
    store_id   INTEGER REFERENCES dim_store(store_id),
    sale_date  DATE,
    quantity   INTEGER,
    revenue    FLOAT,
    cost       FLOAT,
    discount   FLOAT
);


## CELL 4: Dataset — 35 Q&A pairs

In [ ]:
# ── CELL 4: Upload & Load Dataset ───────────────────────────
import json
from google.colab import files

print("Upload your dataset JSON (e.g. retail_text2sql_training_dataset.json)")
uploaded     = files.upload()
DATASET_FILE = list(uploaded.keys())[0]

with open(DATASET_FILE) as f:
    raw = json.load(f)

# Handles two formats:
#   {"TRAINING_DATASET": [...]}  ← wrapped dict
#   [...]                        ← plain list
if isinstance(raw, dict):
    key     = list(raw.keys())[0]
    DATASET = raw[key]
else:
    DATASET = raw

# Normalise id to int so :02d / :3d format specs work everywhere
for i, row in enumerate(DATASET):
    raw_id = row.get("id", i)
    try:
        row["id"] = int(raw_id)
    except (ValueError, TypeError):
        row["id"] = i

# Validate fields
for r in DATASET:
    for field in ["instruction", "sql"]:
        if field not in r:
            print(f"  ⚠  id={r.get('id','?')} missing field: {field}")

# Remove rows missing required fields
DATASET = [r for r in DATASET if "instruction" in r and "sql" in r]

# Pick 12 evenly-spaced test questions across the full dataset
step     = max(1, len(DATASET) // 12)
TEST_SET = DATASET[::step][:12]
TEST_IDS = {r["id"] for r in TEST_SET}

print(f"Dataset loaded    : {len(DATASET)} records")
print(f"Test set          : {len(TEST_SET)} questions")
print(f"Sample instruction: {DATASET[0]['instruction']}")
print(f"Sample SQL        : {DATASET[0]['sql'][:80]}")


Upload your dataset JSON (e.g. retail_text2sql_training_dataset.json)


Saving retail_text2sql_training_dataset(1).json to retail_text2sql_training_dataset(1).json
Dataset loaded    : 500 records
Test set          : 12 questions
Sample instruction: List sales IDs where quantity sold is greater than 1.
Sample SQL        : SELECT sales_id FROM fact_sales WHERE quantity_sold > 1;


## CELL 5: Prompt Template

In [ ]:
# Identical format used for training AND inference.
# phi-2 was trained on Python/text in this style — works well.

PROMPT = """### Schema:
{schema}

### Question:
{question}

### SQL:
"""

def make_prompt(instruction: str) -> str:
    return PROMPT.format(schema=SCHEMA, question=instruction)

def make_training_text(record: dict) -> str:
    """Full training example: prompt + SQL answer."""
    return make_prompt(record["instruction"]) + record["sql"] + "\n"

def extract_sql_from_output(output: str, prompt: str) -> str:
    """Strip the prompt, extract only the SQL."""
    if output.startswith(prompt):
        output = output[len(prompt):]
    output = re.sub(r"```sql|```", "", output).strip()
    output = re.split(r"\n###|\n\n\n|<\|endoftext\|>", output)[0].strip()
    m = re.search(r"(?i)\b(SELECT|WITH)\b", output)
    return output[m.start():].strip() if m else (output if len(output) > 5 else "-- no output")

# Verify
sample = make_training_text(TEST_SET[0])
print(f"Sample training text ({len(sample)//4} tokens):")
print(sample[:300])
print("...")

Sample training text (201 tokens):
### Schema:
CREATE TABLE dim_product (
    product_id   INTEGER PRIMARY KEY,
    product_name VARCHAR(100),
    category     VARCHAR(50),
    brand        VARCHAR(50),
    unit_price   FLOAT
);
CREATE TABLE dim_store (
    store_id     INTEGER PRIMARY KEY,
    store_name   VARCHAR(100),
    city    
...


## CELL 6: Scoring Functions

In [ ]:
def extract_sql_components(sql: str) -> dict:
    u = sql.upper().strip()
    sel = re.search(r"SELECT\s+(.*?)\s+FROM", u, re.DOTALL)
    wh  = re.search(r"WHERE\s+(.*?)(?:GROUP BY|ORDER BY|HAVING|LIMIT|$)", u, re.DOTALL)
    gb  = re.search(r"GROUP BY\s+(.*?)(?:HAVING|ORDER BY|LIMIT|$)", u, re.DOTALL)
    return {
        "tables":    set(re.findall(r"(?:FROM|JOIN)\s+(\w+)", u)),
        "select":    set(re.split(r",\s*", sel.group(1).strip())) if sel else set(),
        "where":     set(re.split(r"\s+AND\s+|\s+OR\s+", wh.group(1).strip())) if wh else set(),
        "group_by":  set(re.split(r",\s*", gb.group(1).strip())) if gb else set(),
        "has_join":  "JOIN" in u,
        "has_agg":   any(k in u for k in ["SUM(","COUNT(","AVG(","MAX(","MIN("]),
        "has_group": "GROUP BY" in u,
        "has_where": "WHERE" in u,
        "has_order": "ORDER BY" in u,
        "has_limit": "LIMIT" in u,
    }

def component_match_score(gold: str, pred: str) -> float:
    g, p = extract_sql_components(gold), extract_sql_components(pred)
    scores = []
    for k in ["tables","select","where","group_by"]:
        gk, pk = g[k], p[k]
        if not gk and not pk: scores.append(1.0)
        elif not gk or not pk: scores.append(0.0)
        else: scores.append(len(gk & pk)/max(len(gk),len(pk)))
    for k in ["has_join","has_agg","has_group","has_where","has_order","has_limit"]:
        scores.append(1.0 if g[k]==p[k] else 0.0)
    return round(sum(scores)/len(scores)*100, 2)

def bleu_score(gold: str, pred: str) -> float:
    try:   return round(sacrebleu.sentence_bleu(pred, [gold]).score, 2)
    except: return 0.0

def run_evaluation(model, tokenizer, test_set: list, label: str) -> dict:
    """Ask all test questions, score results, print comparison."""
    model.eval()
    results = []

    print(f"\n{'='*62}")
    print(f"  {label}")
    print(f"{'='*62}")

    for rec in test_set:
        prompt = make_prompt(rec["instruction"])
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=512,
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        raw     = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        pred    = extract_sql_from_output(raw, prompt)
        gold    = rec["sql"]
        cm      = component_match_score(gold, pred)
        bl      = bleu_score(gold, pred)

        results.append({
            "id":              rec["id"],
            "instruction":     rec["instruction"],
            "gold_sql":        gold,
            "predicted_sql":   pred,
            "component_match": cm,
            "bleu":            bl,
            "combined":        round((cm+bl)/2, 2),
        })

        print(f"\n  Q{rec['id']:02d}: {rec['instruction']}")
        print(f"  GOLD : {gold[:90]}")
        print(f"  PRED : {pred[:90]}")
        print(f"  CM={cm}%  BLEU={bl}")

    avg_cm   = round(sum(r["component_match"] for r in results)/len(results), 2)
    avg_bleu = round(sum(r["bleu"] for r in results)/len(results), 2)
    combined = round((avg_cm+avg_bleu)/2, 2)

    print(f"\n  {'─'*54}")
    print(f"  Avg Component Match : {avg_cm}%")
    print(f"  Avg BLEU Score      : {avg_bleu}%")
    print(f"  COMBINED            : {combined}%")
    print(f"{'='*62}\n")

    return {"component_match":avg_cm, "bleu":avg_bleu,
            "combined":combined, "details":results}

print("Scoring functions ready ✅")

Scoring functions ready ✅


## CELL 7: Load phi-2

In [ ]:
# phi-2 (2.7B) is natively supported in modern transformers.
# No version pinning needed. No trust_remote_code issues.
# fp16 fits in T4 (16GB): ~5.4GB weights + LoRA overhead.

MODEL_NAME = "microsoft/phi-2"
print(f"Loading {MODEL_NAME} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_cache    = False   # required for gradient checkpointing

params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded: {params:.2f}B params ✅")
print(f"pad_token_id : {tokenizer.pad_token_id}")

Loading microsoft/phi-2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: 2.78B params ✅
pad_token_id : 50256


## CELL 8: BEFORE Training Evaluation

In [ ]:
torch.cuda.empty_cache(); gc.collect()
before_results = run_evaluation(model, tokenizer, TEST_SET, "BEFORE TRAINING")
before_score   = before_results["combined"]


  BEFORE TRAINING

  Q01: List sales IDs where quantity sold is greater than 1.
  GOLD : SELECT sales_id FROM fact_sales WHERE quantity_sold > 1;
  PRED : SELECT sale_id FROM fact_sales WHERE quantity > 1;
  CM=80.0%  BLEU=58.74

  Q42: Count customers younger than 25.
  GOLD : SELECT COUNT(*) AS customer_count FROM dim_customer WHERE age < 25;
  PRED : SELECT COUNT(*) FROM customer WHERE age < 25;
  CM=80.0%  BLEU=45.55

  Q83: Show top 5 sales records by highest net sales amount.
  GOLD : SELECT sales_id, net_sales_amount FROM fact_sales ORDER BY net_sales_amount DESC LIMIT 5;
  PRED : SELECT p.product_name, s.store_name, SUM(f.quantity * f.unit_price - f.cost) AS net_sales

  CM=43.33%  BLEU=9.33

  Q124: Count sales rows with profit amount greater than 22.
  GOLD : SELECT COUNT(*) AS sales_count FROM fact_sales WHERE profit_amount > 22;
  PRED : SELECT COUNT(*) FROM fact_sales WHERE revenue - cost > 22;
  CM=80.0%  BLEU=41.84

  Q165: Show total profit from stores in region South.

## CELL 9: Prepare Training Dataset

In [ ]:
hf_dataset = Dataset.from_list([
    {"text": make_training_text(r)} for r in DATASET
])

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )

tokenized = hf_dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Training samples : {len(tokenized)}")
print(f"Sample token count: {len(tokenized[0]['input_ids'])}")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Training samples : 500
Sample token count: 284


## CELL 10: LoRA Config

In [ ]:
# ── FIX: enable gradient checkpointing BEFORE get_peft_model ────────────
# PEFT's get_peft_model() calls enable_input_require_grads() internally,
# but ONLY if it detects GC is already active at that moment.
# If Trainer enables GC later (after wrapping), the hook is never attached
# → lora_A/lora_B get requires_grad=False → no gradients → ~40-byte adapter.
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# phi-2 target modules (confirmed from HuggingFace phi-2 discussions):
# q_proj, k_proj, v_proj = attention projections
# fc1, fc2              = MLP feed-forward layers
# dense                 = attention output projection

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Verification: confirm LoRA was applied correctly ─────────────────────
lora_params = [(n, p) for n, p in model.named_parameters() if "lora_" in n]
trainable   = [p for _, p in lora_params if p.requires_grad]
print(f"\nLoRA tensors found   : {len(lora_params)}")
print(f"Trainable LoRA params: {len(trainable)}")
if not trainable:
    raise RuntimeError("\u274c No LoRA params trainable — gradient_checkpointing_enable() must run before get_peft_model()")
print("\nSample (first 4):")
for name, param in lora_params[:4]:
    print(f"  {name[-60:]:<60}  requires_grad={param.requires_grad}")
print("\n\u2705 LoRA applied correctly — lora_A and lora_B are trainable.")

trainable params: 23,592,960 || all params: 2,803,276,800 || trainable%: 0.8416

LoRA tensors found   : 384
Trainable LoRA params: 384

Sample (first 4):
  .model.model.layers.0.self_attn.q_proj.lora_A.default.weight  requires_grad=True
  .model.model.layers.0.self_attn.q_proj.lora_B.default.weight  requires_grad=True
  .model.model.layers.0.self_attn.k_proj.lora_A.default.weight  requires_grad=True
  .model.model.layers.0.self_attn.k_proj.lora_B.default.weight  requires_grad=True

✅ LoRA applied correctly — lora_A and lora_B are trainable.


## CELL 11: Train

In [ ]:
# Plain HuggingFace Trainer — no TRL dependency, no API version issues.
# DataCollatorForLanguageModeling handles causal LM training automatically.

training_args = TrainingArguments(
    output_dir="./phi2-retail-sql",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    fp16=False,
    logging_steps=10,
    save_strategy="no",
    optim="adamw_torch",
    report_to="none",
    weight_decay=0.01,
    # gradient_checkpointing enabled manually before get_peft_model (see Cell 10)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Starting fine-tuning ...")
print("Expected: loss ~2 at start, converges to ~0.3 by epoch 10\n")
trainer.train()
print("\nFine-tuning complete ✅")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting fine-tuning ...
Expected: loss ~2 at start, converges to ~0.3 by epoch 10



Step,Training Loss
10,0.969050
20,0.947665
30,0.947028
40,0.891737
50,0.810607
60,0.732506
70,0.707929
80,0.590713
90,0.499104
100,0.408339



Fine-tuning complete ✅


## CELL 12: AFTER Training Evaluation

In [ ]:
torch.cuda.empty_cache(); gc.collect()
merged_model  = model.merge_and_unload()
after_results = run_evaluation(merged_model, tokenizer, TEST_SET, "AFTER TRAINING")
after_score   = after_results["combined"]


  AFTER TRAINING

  Q01: List sales IDs where quantity sold is greater than 1.
  GOLD : SELECT sales_id FROM fact_sales WHERE quantity_sold > 1;
  PRED : SELECT s.f.customer_id, f.net_sales_amount FROM fact_sales f INNER JOIN dim_date d ON f.da
  CM=15.0%  BLEU=5.21

  Q42: Count customers younger than 25.
  GOLD : SELECT COUNT(*) AS customer_count FROM dim_customer WHERE age < 25;
  PRED : SELECT COUNT(*) AS customer_count FROM dim_customer WHERE age < 25;
  CM=100.0%  BLEU=100.0

  Q83: Show top 5 sales records by highest net sales amount.
  GOLD : SELECT sales_id, net_sales_amount FROM fact_sales ORDER BY net_sales_amount DESC LIMIT 5;
  PRED : SELECT SUM(f.net_sales_amount) AS total_net_sales FROM fact_sales f INNER JOIN dim_date d 
  CM=45.0%  BLEU=17.73

  Q124: Count sales rows with profit amount greater than 22.
  GOLD : SELECT COUNT(*) AS sales_count FROM fact_sales WHERE profit_amount > 22;
  PRED : SELECT COUNT(*) AS total_net_sales FROM fact_sales f INNER JOIN dim_date d O

## CELL 13: Final Report

In [ ]:
improvement     = round(after_score - before_score, 2)
improvement_pct = round((improvement/before_score)*100, 2) if before_score > 0 else 0

print("\n" + "="*62)
print("         BENCHMARK COMPARISON REPORT")
print("="*62)
print(f"  Model   : {MODEL_NAME} (2.7B, fp16, LoRA r=16)")
print(f"  Schema  : dim_product, dim_store, fact_sales")
print(f"  Training: {len(DATASET)} Q&A pairs, 10 epochs")
print(f"  Test    : {len(TEST_SET)} questions (same before & after)")
print(f"\n  {'Metric':<22} {'BEFORE':>8}  {'AFTER':>8}  {'DELTA':>8}")
print(f"  {'─'*52}")
d_cm   = after_results['component_match'] - before_results['component_match']
d_bleu = after_results['bleu']            - before_results['bleu']
print(f"  {'Component Match':<22} {before_results['component_match']:>7}%  {after_results['component_match']:>7}%  {d_cm:>+7.2f}%")
print(f"  {'BLEU Score':<22} {before_results['bleu']:>7}%  {after_results['bleu']:>7}%  {d_bleu:>+7.2f}%")
print(f"  {'─'*52}")
print(f"  {'COMBINED':<22} {before_score:>7}%  {after_score:>7}%  {improvement:>+7.2f}%")
print(f"  {'Relative Improvement':<22} {improvement_pct:>+8.2f}%")
print("="*62)
if improvement_pct >= 10:
    print(f"\n  ✅ TARGET MET — {improvement_pct}% improvement")
else:
    print(f"\n  ⚠  {improvement_pct}% — try num_train_epochs=15 in Cell 11")
print("="*62)

print("\n── Per-Question Detail ──────────────────────────────────")
bm = {r["id"]: r for r in before_results["details"]}
am = {r["id"]: r for r in after_results["details"]}
print(f"  {'Q':>3}  {'Instruction':<38}  {'BEFORE':>7}  {'AFTER':>7}  {'DELTA':>7}")
print(f"  {'─'*68}")
for qid in sorted(TEST_IDS):
    b = bm[qid]["combined"]
    a = am[qid]["combined"]
    ins = bm[qid]["instruction"][:36] + ".."
    print(f"  {qid:3d}  {ins:<38}  {b:>6.1f}%  {a:>6.1f}%  {a-b:>+6.1f}%")


         BENCHMARK COMPARISON REPORT
  Model   : microsoft/phi-2 (2.7B, fp16, LoRA r=16)
  Schema  : dim_product, dim_store, fact_sales
  Training: 500 Q&A pairs, 10 epochs
  Test    : 12 questions (same before & after)

  Metric                   BEFORE     AFTER     DELTA
  ────────────────────────────────────────────────────
  Component Match           64.3%     72.5%    +8.20%
  BLEU Score                30.6%     58.3%   +27.70%
  ────────────────────────────────────────────────────
  COMBINED                 47.45%     65.4%   +17.95%
  Relative Improvement     +37.83%

  ✅ TARGET MET — 37.83% improvement

── Per-Question Detail ──────────────────────────────────
    Q  Instruction                              BEFORE    AFTER    DELTA
  ────────────────────────────────────────────────────────────────────
    1  List sales IDs where quantity sold i..    69.4%    10.1%   -59.3%
   42  Count customers younger than 25...        62.8%   100.0%   +37.2%
   83  Show top 5 sales records

## CELL 14: Download Results

In [ ]:
import json
from google.colab import files as colab_files

report = {
    "model": MODEL_NAME,
    "training_samples": len(DATASET),
    "test_questions": len(TEST_SET),
    "before": before_results,
    "after":  after_results,
    "improvement_absolute": improvement,
    "improvement_pct": improvement_pct,
}
with open("benchmark_report.json", "w") as f:
    json.dump(report, f, indent=2)
colab_files.download("benchmark_report.json")
print("Downloaded: benchmark_report.json ✅")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: benchmark_report.json ✅


In [ ]:
import torch

def generate_sql(instruction):
    """
    Stand-alone inference function for the fine-tuned Phi-2 model.
    Uses the exact prompt format from the training phase.
    """
    # Use the prompt template defined in Cell 5 of your notebook
    prompt = make_prompt(instruction)

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    # Generate output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,        # Greedy decoding for stable SQL
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode and clean the output using your notebook's extraction logic
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql_only = extract_sql_from_output(raw_output, prompt)

    return sql_only

# --- TEST BLOCK ---
# Test 1: Calculation Logic
q1 = "List all product names and their profit margins, where profit margin is (revenue - cost) / revenue."
print(f"Instruction: {q1}\nSQL: {generate_sql(q1)}\n")

# Test 2: Multi-Condition Filtering
q2 = "Find the total revenue for 'Apparel' products sold in 'flagship' stores with quantity > 5."
print(f"Instruction: {q2}\nSQL: {generate_sql(q2)}\n")

# Test 3: Negative Filtering (Your previous error)
q3 = "Show the total quantity of products sold in every region except for the 'North' region."
print(f"Instruction: {q3}\nSQL: {generate_sql(q3)}")

Instruction: List all product names and their profit margins, where profit margin is (revenue - cost) / revenue.
SQL: SELECT s.loyalty_tier, f.product_id, p.product_name, f.profit_amount FROM fact_sales f INNER JOIN dim_customer c ON f.customer_id = c.customer_id GROUP BY c.loyalty_tier HAVING SUM(f.net_sales_amount) > 100 ORDER BY total_net_sales DESC;

Instruction: Find the total revenue for 'Apparel' products sold in 'flagship' stores with quantity > 5.
SQL: SELECT SUM(f.net_sales_amount) AS total_net_sales FROM fact_sales f INNER JOIN dim_store s ON f.store_id = s.store_id GROUP BY s.store_id HAVING SUM(f.net_sales_amount) > 5 ORDER BY total_net_sales DESC;

Instruction: Show the total quantity of products sold in every region except for the 'North' region.
SQL: SELECT s.region_name, SUM(f.quantity_sold) AS total_quantity_sold FROM fact_sales f INNER JOIN dim_store s ON f.store_id = s.store_id GROUP BY s.region_name HAVING SUM(f.quantity_sold) > 100 ORDER BY total_net_sales DESC;


In [ ]:
import torch
from transformers import AutoModelForCausalLM

print("Loading raw base model for comparison (this may take a minute)...")
# Load a fresh, untrained instance of Phi-2
raw_base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

def generate_sql_BEFORE(instruction):
    prompt = make_prompt(instruction)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(raw_base_model.device)

    with torch.no_grad():
        outputs = raw_base_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return extract_sql_from_output(raw_output, prompt)

# --- THE REAL COMPARISON ---
test_questions = [
    "List all product names and their profit margins, where profit margin is (revenue - cost) / revenue.",
    "Find the total revenue for 'Apparel' products sold in 'flagship' stores with quantity > 5.",
    "Show the total quantity of products sold in every region except for the 'North' region."
]

for i, q in enumerate(test_questions, 1):
    print(f"\nTEST {i}: {q}")
    print(f"BEFORE TRAINING (Raw Phi-2):\n{generate_sql_BEFORE(q)}")
    print(f"AFTER TRAINING (Your Fine-tuned Model):\n{generate_sql(q)}")
    print("-" * 60)

Loading raw base model for comparison (this may take a minute)...


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]


TEST 1: List all product names and their profit margins, where profit margin is (revenue - cost) / revenue.
BEFORE TRAINING (Raw Phi-2):
SELECT product_name, (revenue - cost) / revenue AS profit_margin
FROM dim_product
JOIN fact_sales ON dim_product.product_id = fact_sales.product_id
WHERE revenue > cost;
AFTER TRAINING (Your Fine-tuned Model):
SELECT s.loyalty_tier, f.product_id, p.product_name, f.profit_amount FROM fact_sales f INNER JOIN dim_customer c ON f.customer_id = c.customer_id GROUP BY c.loyalty_tier HAVING SUM(f.net_sales_amount) > 100 ORDER BY total_net_sales DESC;
------------------------------------------------------------

TEST 2: Find the total revenue for 'Apparel' products sold in 'flagship' stores with quantity > 5.
BEFORE TRAINING (Raw Phi-2):
SELECT SUM(revenue) FROM fact_sales
WHERE product_name LIKE '%Apparel%' AND store_format = 'flagship' AND quantity > 5;
AFTER TRAINING (Your Fine-tuned Model):
SELECT SUM(f.net_sales_amount) AS total_net_sales FROM fact_sale

In [ ]:
model.save_pretrained("my_modelt")
tokenizer.save_pretrained("my_modelt")

('my_modelt/tokenizer_config.json', 'my_modelt/tokenizer.json')

In [ ]:
import shutil
shutil.make_archive("modelt", 'zip', "my_modelt")

'/content/modelt.zip'

In [ ]:
from google.colab import files
files.download("modelt.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "adapter" in file:
            print(os.path.join(root, file))

/content/my_modelt/adapter_model.safetensors
/content/my_modelt/adapter_config.json


In [ ]:
from google.colab import files
files.download("/content/my_modelt/adapter_model.safetensors")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

path = "/content/my_modelt/adapter_model.safetensors"
print("Size:", os.path.getsize(path), "bytes")

Size: 40 bytes


In [ ]:
for f in os.listdir("/content/my_modelt"):
    p = os.path.join("/content/my_modelt", f)
    print(f, os.path.getsize(p)/1024, "KB")

README.md 5.0625 KB
tokenizer.json 3481.494140625 KB
tokenizer_config.json 0.310546875 KB
adapter_model.safetensors 0.0390625 KB
adapter_config.json 1.041015625 KB


In [ ]:
dir()

['AutoModelForCausalLM',
 'AutoTokenizer',
 'DATASET',
 'DATASET_FILE',
 'DataCollatorForLanguageModeling',
 'Dataset',
 'In',
 'LoraConfig',
 'MODEL_NAME',
 'Out',
 'PROMPT',
 'SCHEMA',
 'TEST_IDS',
 'TEST_SET',
 'TaskType',
 'Trainer',
 'TrainingArguments',
 '_',
 '_16',
 '_17',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '_dh',
 '_i',
 '_i1',
 '_i10',
 '_i11',
 '_i12',
 '_i13',
 '_i14',
 '_i15',
 '_i16',
 '_i17',
 '_i18',
 '_i19',
 '_i2',
 '_i20',
 '_i21',
 '_i22',
 '_i23',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_i7',
 '_i8',
 '_i9',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'a',
 'after_results',
 'after_score',
 'am',
 'b',
 'before_results',
 'before_score',
 'bleu_score',
 'bm',
 'colab_files',
 'component_match_score',
 'd_bleu',
 'd_cm',
 'dirs',
 'exit',
 'extract_sql_components',
 'extract_sql_from_output',
 'f',
 'field',
 'file',
 'files',
 'gc',
 'generate_sql',
 'generate_sql_BEFORE',
 'get_ipython',
 'get_peft

In [ ]:
print(type(model))

<class 'peft.peft_model.PeftModelForCausalLM'>


In [ ]:
print(len(tokenized))

500


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)




In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # VERY IMPORTANT (causal LM)
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator
)

In [ ]:
trainer.train()

Step,Training Loss
10,0.108923
20,0.135924
30,0.121296
40,0.142106
50,0.111677
60,0.115957
70,0.123450
80,0.142104
90,0.099257
100,0.099693


TrainOutput(global_step=1250, training_loss=0.1206910246372223, metrics={'train_runtime': 2251.178, 'train_samples_per_second': 2.221, 'train_steps_per_second': 0.555, 'total_flos': 2.79334258538496e+16, 'train_loss': 0.1206910246372223, 'epoch': 10.0})

In [ ]:
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        print(name)